In [1]:
import pandas as pd 
import numpy as np 

P = "processed"
X_tr = pd.read_csv(f"{P}/X_global_train.csv")
X_te = pd.read_csv(f"{P}/X_global_test.csv")
Y_tr = pd.read_csv(f"{P}/Y_global_train.csv")
Y_te = pd.read_csv(f"{P}/Y_global_test.csv")
info_tr = pd.read_csv(f"{P}/global_train_info.csv")
info_te = pd.read_csv(f"{P}/global_test_info.csv")

print(X_tr.shape, Y_tr.shape, info_tr.shape)
print(info_tr["City"].value_counts())

(39168, 6) (39168, 3) (39168, 2)
City
Brasilia    6528
Cairo       6528
Dubai       6528
London      6528
New York    6528
Sydney      6528
Name: count, dtype: int64


In [2]:
city = "London"

m_tr = info_tr["City"] == city 
m_te = info_tr["City"] == city

X_tr_c, Y_tr_c = X_tr[m_tr], Y_tr[m_tr]
X_te_c, Y_te_c = X_te[m_te], Y_te[m_te]

print(X_tr_c.shape, Y_tr_c.shape)
print(X_te_c.shape, Y_te_c.shape)

(6528, 6) (6528, 3)
(2184, 6) (2184, 3)


/var/folders/cf/qtyslrd93l3d05ncps8f7dj80000gq/T/ipykernel_81302/840737723.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_te_c, Y_te_c = X_te[m_te], Y_te[m_te]
/var/folders/cf/qtyslrd93l3d05ncps8f7dj80000gq/T/ipykernel_81302/840737723.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_te_c, Y_te_c = X_te[m_te], Y_te[m_te]


In [7]:
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

target = "PM25_1hr_ahead"
y_tr = Y_tr_c[target]
y_te = Y_te_c[target]

def score(y_true, y_pred):
    return {"r2": r2_score(y_true, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
            "mae": mean_absolute_error(y_true, y_pred)}
    
lr = LinearRegression().fit(X_tr_c, y_tr)
print("LR:         ", score(y_te, lr.predict(X_te_c)))

# persistence: "PM2.5 an hour from now = PM2.5 right now"
print("Persistence:", score(y_te, X_te_c["PM2.5"]))


LR:          {'r2': 0.918205437936369, 'rmse': np.float64(1.128392204241788), 'mae': 0.790507688871288}
Persistence: {'r2': 0.9157637785705023, 'rmse': np.float64(1.145110244704619), 'mae': 0.7908882783882785}


In [6]:
print(dict(zip(X_tr_c.columns, lr.coef_.round(3))))

{'CO': np.float64(0.006), 'NO2': np.float64(0.083), 'SO2': np.float64(-0.012), 'O3': np.float64(0.018), 'PM2.5': np.float64(0.391), 'PM10': np.float64(0.027)}


In [8]:
from sklearn.ensemble import RandomForestRegressor
import time

In [ ]:
HORIZONS = [1, 6, 24]
DATASETS = {"global": "City", "beijing": "station"}

def load(name):
    P = "processed"
    return (pd.read_csv(f"{P}/X_{name}_train.csv"), pd.read_csv(f"{P}/X_{name}_test.csv"),
            pd.read_csv(f"{P}/Y_{name}_train.csv"), pd.read_csv(f"{P}/Y_{name}_test.csv"),
            pd.read_csv(f"{P}/{name}_train_info.csv"), pd.read_csv(f"{P}/{name}_test_info.csv"))

rows = []
t0 = time.time()
for name, loc_col in DATASETS.items():
    X_tr, X_te, Y_tr, Y_te, info_tr, info_te = load(name)
    for loc in sorted(info_tr[loc_col].unique()):
        m_tr, m_te = info_tr[loc_col] == loc, info_te[loc_col] == loc
        Xtr, Xte = X_tr[m_tr], X_te[m_te]
        for h in HORIZONS:
            t = f"PM25_{h}hr_ahead"
            ytr, yte = Y_tr.loc[m_tr, t], Y_te.loc[m_te, t]

            SEEDS = [0, 1, 2, 3, 4]
            preds = {
                ("Mean", 0): np.full(len(yte), ytr.mean()),
                ("Persistence", 0): Xte["PM2.5"].values,
                ("Linear Regression", 0): LinearRegression().fit(Xtr, ytr).predict(Xte),
            }
            for s in SEEDS:
                rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=5,
                                           max_features=0.5, n_jobs=-1, random_state=s)
                preds[("Random Forest", s)] = rf.fit(Xtr, ytr).predict(Xte)

            for (model, seed), p in preds.items():
                rows.append({"dataset": name, "location": loc, "horizon": h,
                             "model": model, "seed": seed, **score(yte, p)})
        print(f"[{name}] {loc:14s} done ({time.time() - t0:.0f}s)")

results = pd.DataFrame(rows)
results.to_csv("baseline_results.csv", index=False)

[global] Brasilia       done (0s)
[global] Cairo          done (1s)
[global] Dubai          done (1s)
[global] London         done (1s)
[global] New York       done (2s)
[global] Sydney         done (2s)
[beijing] Aotizhongxin   done (4s)
[beijing] Changping      done (5s)
[beijing] Dingling       done (6s)
[beijing] Dongsi         done (8s)
[beijing] Guanyuan       done (9s)
[beijing] Gucheng        done (11s)
[beijing] Huairou        done (12s)
[beijing] Nongzhanguan   done (14s)
[beijing] Shunyi         done (15s)
[beijing] Tiantan        done (17s)
[beijing] Wanliu         done (18s)
[beijing] Wanshouxigong  done (20s)


In [10]:
summary = (results.groupby(["dataset", "horizon", "model"])[["r2", "rmse", "mae"]]
                  .mean().round(3))
summary

r2    rmse     mae
dataset horizon model                                   
beijing 1       Linear Regression  0.948  18.565  10.278
                Mean              -0.002  81.799  59.592
                Persistence        0.946  18.989  10.104
                Random Forest      0.948  18.566  10.127
        6       Linear Regression  0.622  50.331  31.409
                Mean              -0.002  81.854  59.605
                Persistence        0.562  54.174  31.742
                Random Forest      0.619  50.508  30.620
        24      Linear Regression  0.233  71.458  50.492
                Mean              -0.003  81.753  59.652
                Persistence       -0.144  87.215  57.570
                Random Forest      0.206  72.655  50.238
global  1       Linear Regression  0.897   2.410   1.586
                Mean              -0.082   8.119   6.590
                Persistence        0.896   2.402   1.534
                Random Forest      0.890   2.505   1.657
        6       Linear Regression  0.225   6.657   5.009
                Mean              -0.083   8.124   6.595
                Persistence        0.074   7.341   5.318
                Random Forest      0.234   6.782   5.039
        24      Linear Regression  0.036   7.419   5.630
                Mean              -0.086   8.146   6.617
                Persistence       -0.202   8.215   5.884
                Random Forest     -0.024   7.718   5.971